<a href="https://colab.research.google.com/github/va4756/bigdata_raejung/blob/main/LLM_in_Production.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DeepLearning_pytorch 폴더 - llm-production에 데이터 있음

# 2장 LLM에서의 언어 모델링

## 2-1 생성형 N-gram

In [55]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [56]:
# NLTK + PlaintextCorpusReader (Colab용)
import os
import nltk

from nltk.corpus.reader import PlaintextCorpusReader
from nltk.util import everygrams
from nltk.lm.preprocessing import (
    pad_both_ends, flatten, padded_everygram_pipeline
)
from nltk.lm import MLE

In [57]:
# NLTK tokenizer 다운로드
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [58]:
# txt 파일 업로드
from google.colab import files
uploaded = files.upload()

In [59]:
# 현재 업로드된 txt 파일 확인
print("현재 폴더 파일 목록:")
print(os.listdir("./"))

현재 폴더 파일 목록:
['.config', 'drive', 'sample_data']


In [60]:
# Corpus 생성
# 현재 디렉토리의 모든 txt 파일 사용
my_corpus = PlaintextCorpusReader("./", r".*\.txt")

In [61]:
# corpus 안의 파일 확인
print("\nCorpus files:")
print(my_corpus.fileids())


Corpus files:
['drive/.shortcut-targets-by-id/1_XuoiGdAoueqwYcaJtt5NhJEFGHJKtvG/data/CH07/CH07/annotations/list.txt', 'drive/.shortcut-targets-by-id/1_XuoiGdAoueqwYcaJtt5NhJEFGHJKtvG/data/CH07/CH07/annotations/test.txt', 'drive/.shortcut-targets-by-id/1_XuoiGdAoueqwYcaJtt5NhJEFGHJKtvG/data/CH07/CH07/annotations/trainval.txt', 'drive/.shortcut-targets-by-id/1_XuoiGdAoueqwYcaJtt5NhJEFGHJKtvG/data/CH11.txt', 'drive/MyDrive/SentiWord_Dict.txt', 'drive/MyDrive/kaggle_dataset/HousePrices.txt', 'drive/MyDrive/nmt-master/nmt-master/nmt/testdata/test_embed.txt', 'drive/MyDrive/nmt-master/nmt-master/nmt/testdata/test_embed_with_header.txt']


In [62]:
# 특정 파일 선택
# 업로드한 txt 파일 이름 사용
file_id = my_corpus.fileids()[0]
print(f"\n선택된 파일: {file_id}")


선택된 파일: drive/.shortcut-targets-by-id/1_XuoiGdAoueqwYcaJtt5NhJEFGHJKtvG/data/CH07/CH07/annotations/list.txt


In [63]:
# 문장 출력
for sent in my_corpus.sents(fileids=file_id):
    print(sent)

['#', 'Image', 'CLASS', '-', 'ID', 'SPECIES', 'BREED', 'ID', '#', 'ID', ':', '1', ':', '37', 'Class', 'ids', '#', 'SPECIES', ':', '1', ':', 'Cat', '2', ':', 'Dog', '#', 'BREED', 'ID', ':', '1', '-', '25', ':', 'Cat', '1', ':', '12', ':', 'Dog', '#', 'All', 'images', 'with', '1st', 'letter', 'as', 'captial', 'are', 'cat', 'images', '#', 'images', 'with', 'small', 'first', 'letter', 'are', 'dog', 'images', 'Abyssinian_100', '1', '1', '1', 'Abyssinian_101', '1', '1', '1', 'Abyssinian_102', '1', '1', '1', 'Abyssinian_103', '1', '1', '1', 'Abyssinian_104', '1', '1', '1', 'Abyssinian_105', '1', '1', '1', 'Abyssinian_106', '1', '1', '1', 'Abyssinian_107', '1', '1', '1', 'Abyssinian_108', '1', '1', '1', 'Abyssinian_109', '1', '1', '1', 'Abyssinian_10', '1', '1', '1', 'Abyssinian_110', '1', '1', '1', 'Abyssinian_111', '1', '1', '1', 'Abyssinian_112', '1', '1', '1', 'Abyssinian_113', '1', '1', '1', 'Abyssinian_114', '1', '1', '1', 'Abyssinian_115', '1', '1', '1', 'Abyssinian_116', '1', '1', '1',

In [64]:
# 문장 확인
sents = list(my_corpus.sents(fileids=file_id))
print(f"\n총 문장 수: {len(sents)}")


총 문장 수: 1


In [65]:
"""
핵심 수정 사항:

"./data/hamlet.txt" → Colab에서는 업로드 후 현재 폴더(./)에 저장됨
"fileids" 는 전체 경로가 아니라 파일명만 사용
r".*\.txt" raw string 사용
punkt, punkt_tab 다운로드 추가
files.upload() 로 txt 파일 업로드 가능

예:

hamlet.txt 업로드
자동으로 corpus에 등록
문장 단위 토큰화 출력됨
"""

<>:6: SyntaxWarning: invalid escape sequence '\.'
<>:6: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_4426/3644539359.py:6: SyntaxWarning: invalid escape sequence '\.'
  r".*\.txt" raw string 사용


'\n핵심 수정 사항:\n\n"./data/hamlet.txt" → Colab에서는 업로드 후 현재 폴더(./)에 저장됨\n"fileids" 는 전체 경로가 아니라 파일명만 사용\nr".*\\.txt" raw string 사용\npunkt, punkt_tab 다운로드 추가\nfiles.upload() 로 txt 파일 업로드 가능\n\n예:\n\nhamlet.txt 업로드\n자동으로 corpus에 등록\n문장 단위 토큰화 출력됨\n'

In [66]:
# 예제 문장 선택
# 문장 개수보다 작은 안전한 인덱스 사용
idx = min(1104, len(sents)-1)
print("\n선택 문장:")
print(sents[idx])


선택 문장:
['#', 'Image', 'CLASS', '-', 'ID', 'SPECIES', 'BREED', 'ID', '#', 'ID', ':', '1', ':', '37', 'Class', 'ids', '#', 'SPECIES', ':', '1', ':', 'Cat', '2', ':', 'Dog', '#', 'BREED', 'ID', ':', '1', '-', '25', ':', 'Cat', '1', ':', '12', ':', 'Dog', '#', 'All', 'images', 'with', '1st', 'letter', 'as', 'captial', 'are', 'cat', 'images', '#', 'images', 'with', 'small', 'first', 'letter', 'are', 'dog', 'images', 'Abyssinian_100', '1', '1', '1', 'Abyssinian_101', '1', '1', '1', 'Abyssinian_102', '1', '1', '1', 'Abyssinian_103', '1', '1', '1', 'Abyssinian_104', '1', '1', '1', 'Abyssinian_105', '1', '1', '1', 'Abyssinian_106', '1', '1', '1', 'Abyssinian_107', '1', '1', '1', 'Abyssinian_108', '1', '1', '1', 'Abyssinian_109', '1', '1', '1', 'Abyssinian_10', '1', '1', '1', 'Abyssinian_110', '1', '1', '1', 'Abyssinian_111', '1', '1', '1', 'Abyssinian_112', '1', '1', '1', 'Abyssinian_113', '1', '1', '1', 'Abyssinian_114', '1', '1', '1', 'Abyssinian_115', '1', '1', '1', 'Abyssinian_116', '1', '

In [67]:
# Padding + Everygrams
padded_trigrams = list(pad_both_ends(sents[idx], n=2))

In [68]:
print("\nPadding 결과:")
print(padded_trigrams)


Padding 결과:
['<s>', '#', 'Image', 'CLASS', '-', 'ID', 'SPECIES', 'BREED', 'ID', '#', 'ID', ':', '1', ':', '37', 'Class', 'ids', '#', 'SPECIES', ':', '1', ':', 'Cat', '2', ':', 'Dog', '#', 'BREED', 'ID', ':', '1', '-', '25', ':', 'Cat', '1', ':', '12', ':', 'Dog', '#', 'All', 'images', 'with', '1st', 'letter', 'as', 'captial', 'are', 'cat', 'images', '#', 'images', 'with', 'small', 'first', 'letter', 'are', 'dog', 'images', 'Abyssinian_100', '1', '1', '1', 'Abyssinian_101', '1', '1', '1', 'Abyssinian_102', '1', '1', '1', 'Abyssinian_103', '1', '1', '1', 'Abyssinian_104', '1', '1', '1', 'Abyssinian_105', '1', '1', '1', 'Abyssinian_106', '1', '1', '1', 'Abyssinian_107', '1', '1', '1', 'Abyssinian_108', '1', '1', '1', 'Abyssinian_109', '1', '1', '1', 'Abyssinian_10', '1', '1', '1', 'Abyssinian_110', '1', '1', '1', 'Abyssinian_111', '1', '1', '1', 'Abyssinian_112', '1', '1', '1', 'Abyssinian_113', '1', '1', '1', 'Abyssinian_114', '1', '1', '1', 'Abyssinian_115', '1', '1', '1', 'Abyssinian_

In [69]:
print("\nEverygrams 결과:")
print(list(everygrams(padded_trigrams, max_len=3)))


Everygrams 결과:
[('<s>',), ('<s>', '#'), ('<s>', '#', 'Image'), ('#',), ('#', 'Image'), ('#', 'Image', 'CLASS'), ('Image',), ('Image', 'CLASS'), ('Image', 'CLASS', '-'), ('CLASS',), ('CLASS', '-'), ('CLASS', '-', 'ID'), ('-',), ('-', 'ID'), ('-', 'ID', 'SPECIES'), ('ID',), ('ID', 'SPECIES'), ('ID', 'SPECIES', 'BREED'), ('SPECIES',), ('SPECIES', 'BREED'), ('SPECIES', 'BREED', 'ID'), ('BREED',), ('BREED', 'ID'), ('BREED', 'ID', '#'), ('ID',), ('ID', '#'), ('ID', '#', 'ID'), ('#',), ('#', 'ID'), ('#', 'ID', ':'), ('ID',), ('ID', ':'), ('ID', ':', '1'), (':',), (':', '1'), (':', '1', ':'), ('1',), ('1', ':'), ('1', ':', '37'), (':',), (':', '37'), (':', '37', 'Class'), ('37',), ('37', 'Class'), ('37', 'Class', 'ids'), ('Class',), ('Class', 'ids'), ('Class', 'ids', '#'), ('ids',), ('ids', '#'), ('ids', '#', 'SPECIES'), ('#',), ('#', 'SPECIES'), ('#', 'SPECIES', ':'), ('SPECIES',), ('SPECIES', ':'), ('SPECIES', ':', '1'), (':',), (':', '1'), (':', '1', ':'), ('1',), ('1', ':'), ('1', ':', 'C

In [70]:
# Flatten Example
flattened = list(
    flatten(pad_both_ends(sent, n=2) for sent in sents)
)
print("\nFlattened tokens 일부:")
print(flattened[:50])


Flattened tokens 일부:
['<s>', '#', 'Image', 'CLASS', '-', 'ID', 'SPECIES', 'BREED', 'ID', '#', 'ID', ':', '1', ':', '37', 'Class', 'ids', '#', 'SPECIES', ':', '1', ':', 'Cat', '2', ':', 'Dog', '#', 'BREED', 'ID', ':', '1', '-', '25', ':', 'Cat', '1', ':', '12', ':', 'Dog', '#', 'All', 'images', 'with', '1st', 'letter', 'as', 'captial', 'are', 'cat']


In [71]:
# N-Gram 학습 데이터 생성
train, vocab = padded_everygram_pipeline(3, sents)

In [72]:
# MLE 3-Gram 모델 생성
lm = MLE(3)
print("\n초기 vocab 크기:")
print(len(lm.vocab))


초기 vocab 크기:
0


In [73]:
# 모델 학습
lm.fit(train, vocab)
print("\n학습 후 vocab 크기:")
print(len(lm.vocab))

print("\nVocabulary 일부:")
print(list(lm.vocab)[:30])


학습 후 vocab 크기:
7413

Vocabulary 일부:
['<s>', '#', 'Image', 'CLASS', '-', 'ID', 'SPECIES', 'BREED', ':', '1', '37', 'Class', 'ids', 'Cat', '2', 'Dog', '25', '12', 'All', 'images', 'with', '1st', 'letter', 'as', 'captial', 'are', 'cat', 'small', 'first', 'dog']


In [74]:
# 문장 생성
print("\nGenerated text:")
print(lm.generate(6, ["to", "be"]))


Generated text:
['10', 'Russian_Blue_8', '28', '1', '10', 'Russian_Blue_205']


In [75]:
# Vocabulary Lookup
print("\nKnown words lookup:")
print(list(lm.vocab.lookup(sents[idx])))
print("\nUnknown words lookup:")
print(list(lm.vocab.lookup(["aliens", "from", "Mars"])))


Known words lookup:
['#', 'Image', 'CLASS', '-', 'ID', 'SPECIES', 'BREED', 'ID', '#', 'ID', ':', '1', ':', '37', 'Class', 'ids', '#', 'SPECIES', ':', '1', ':', 'Cat', '2', ':', 'Dog', '#', 'BREED', 'ID', ':', '1', '-', '25', ':', 'Cat', '1', ':', '12', ':', 'Dog', '#', 'All', 'images', 'with', '1st', 'letter', 'as', 'captial', 'are', 'cat', 'images', '#', 'images', 'with', 'small', 'first', 'letter', 'are', 'dog', 'images', 'Abyssinian_100', '1', '1', '1', 'Abyssinian_101', '1', '1', '1', 'Abyssinian_102', '1', '1', '1', 'Abyssinian_103', '1', '1', '1', 'Abyssinian_104', '1', '1', '1', 'Abyssinian_105', '1', '1', '1', 'Abyssinian_106', '1', '1', '1', 'Abyssinian_107', '1', '1', '1', 'Abyssinian_108', '1', '1', '1', 'Abyssinian_109', '1', '1', '1', 'Abyssinian_10', '1', '1', '1', 'Abyssinian_110', '1', '1', '1', 'Abyssinian_111', '1', '1', '1', 'Abyssinian_112', '1', '1', '1', 'Abyssinian_113', '1', '1', '1', 'Abyssinian_114', '1', '1', '1', 'Abyssinian_115', '1', '1', '1', 'Abyssinian

In [76]:
# N-Gram Count
print("\nCounts:")
print(lm.counts)

print("\nCount of 'be' after 'to':")
print(lm.counts[["to"]]["be"])


Counts:
<NgramCounter with 3 ngram orders and 88374 ngrams>

Count of 'be' after 'to':
0


In [77]:
# Probability Scores
print("\nScores:")
print("P(be) =", lm.score("be"))
print("P(be | to) =", lm.score("be", ["to"]))
print("P(be | not to) =", lm.score("be", ["not", "to"]))


Scores:
P(be) = 0.0
P(be | to) = 0
P(be | not to) = 0


In [78]:
# Log Scores
print("\nLog Scores:")
print("log P(be) =", lm.logscore("be"))
print("log P(be | to) =", lm.logscore("be", ["to"]))
print("log P(be | not to) =", lm.logscore("be", ["not", "to"]))


Log Scores:
log P(be) = -inf
log P(be | to) = -inf
log P(be | not to) = -inf


In [79]:
# Entropy & Perplexity
test = [("to", "be"), ("or", "not"), ("to", "be")]

print("\nEntropy:")
print(lm.entropy(test))

print("\nPerplexity:")
print(lm.perplexity(test))


Entropy:
inf

Perplexity:
inf


## 2-2 범주형 단순 베이즈 모델

In [80]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [81]:
# Naive Bayes with NLTK (Colab Version)
import re
import string
import numpy as np
import nltk

from google.colab import files
from nltk.corpus.reader import PlaintextCorpusReader
from nltk.tokenize import word_tokenize

In [82]:
# NLTK 데이터 다운로드
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [83]:
# txt 파일 업로드
uploaded = files.upload()

In [84]:
# Corpus 생성
my_corpus = PlaintextCorpusReader("./", r".*\.txt")

print("Corpus files:")
print(my_corpus.fileids())

Corpus files:
['drive/.shortcut-targets-by-id/1_XuoiGdAoueqwYcaJtt5NhJEFGHJKtvG/data/CH07/CH07/annotations/list.txt', 'drive/.shortcut-targets-by-id/1_XuoiGdAoueqwYcaJtt5NhJEFGHJKtvG/data/CH07/CH07/annotations/test.txt', 'drive/.shortcut-targets-by-id/1_XuoiGdAoueqwYcaJtt5NhJEFGHJKtvG/data/CH07/CH07/annotations/trainval.txt', 'drive/.shortcut-targets-by-id/1_XuoiGdAoueqwYcaJtt5NhJEFGHJKtvG/data/CH11.txt', 'drive/MyDrive/SentiWord_Dict.txt', 'drive/MyDrive/kaggle_dataset/HousePrices.txt', 'drive/MyDrive/nmt-master/nmt-master/nmt/testdata/test_embed.txt', 'drive/MyDrive/nmt-master/nmt-master/nmt/testdata/test_embed_with_header.txt']


In [85]:
# 첫 번째 txt 파일 사용
file_id = my_corpus.fileids()[0]
print(f"\n사용 파일: {file_id}")


사용 파일: drive/.shortcut-targets-by-id/1_XuoiGdAoueqwYcaJtt5NhJEFGHJKtvG/data/CH07/CH07/annotations/list.txt


In [86]:
# 문장 불러오기
sents = list(my_corpus.sents(fileids=file_id))
print(f"\n총 문장 수: {len(sents)}")


총 문장 수: 1


In [87]:
# utils.py 대신 직접 구현
def process_utt(utt):
    # 문장 전처리

    # 소문자 변환
    utt = utt.lower()

    # 토큰화
    words = word_tokenize(utt)

    # punctuation 제거
    clean_words = []

    for word in words:
        if word not in string.punctuation:
            clean_words.append(word)
        return clean_words

In [88]:
def lookup(freqs, word, label):
    # 단어 빈도 조회
    return freqs.get((word, label), 0)

In [90]:
# Count Utterances
import re
import string

from nltk.stem import PorterStemmer
from nltk.tokenize import TweetTokenizer

def process_utt(utt):
    """
    Input:
        utt: a string containing a utt
    Output:
        utts_clean: a list of words containing the processed utt
    """
    stemmer = PorterStemmer()
    # stopwords_english = stopwords.words('english')
    # remove stock market tickers like $GE
    utt = re.sub(r"\$\w*", "", utt)
    # remove old style utt text "RT"
    utt = re.sub(r"^RT[\s]+", "", utt)
    # remove hyperlinks
    utt = re.sub(r"https?:\/\/.*[\r\n]*", "", utt)
    # remove hashtags
    # only removing the hash # sign from the word
    utt = re.sub(r"#", "", utt)
    # tokenize utts
    tokenizer = TweetTokenizer(
        preserve_case=False, strip_handles=True, reduce_len=True
    )
    utt_tokens = tokenizer.tokenize(utt)

    utts_clean = []
    for word in utt_tokens:
        if word not in string.punctuation:  # remove punctuation
            # utts_clean.append(word)
            stem_word = stemmer.stem(word)  # stemming word
            utts_clean.append(stem_word)

    return utts_clean

In [91]:
def count_utts(result, utts, ys):
    """
    Input:
        result: a dictionary that is used to map each pair to its frequency
        utts: a list of utts
        ys: a list of the sentiment of each utt (either 0 or 1)
    Output:
        result: a dictionary mapping each pair to its frequency
    """
    for y, utt in zip(ys, utts):
        for word in process_utt(utt):
            # define the key, which is the word and label tuple
            pair = (word, y)

            # if the key exists in the dictionary, increment the count
            if pair in result:
                result[pair] += 1

            # if the key is new, add it to the dict and set the count to 1
            else:
                result[pair] = 1
        return result

In [94]:
result = {}
utts = [" ".join(sent) for sent in sents]
ys = [sent.count("be") > 0 for sent in sents]
count_utts(result, utts, ys)

{('imag', False): 5,
 ('class', False): 2,
 ('id', False): 5,
 ('speci', False): 2,
 ('breed', False): 2,
 ('1', False): 2971,
 ('37', False): 201,
 ('cat', False): 3,
 ('2', False): 5579,
 ('dog', False): 3,
 ('25', False): 401,
 ('12', False): 591,
 ('all', False): 1,
 ('with', False): 2,
 ('1st', False): 1,
 ('letter', False): 2,
 ('as', False): 1,
 ('captial', False): 1,
 ('are', False): 2,
 ('small', False): 1,
 ('first', False): 1,
 ('abyssinian', False): 198,
 ('_100', False): 36,
 ('_101', False): 36,
 ('_102', False): 36,
 ('_103', False): 37,
 ('_104', False): 37,
 ('_105', False): 35,
 ('_106', False): 35,
 ('_107', False): 36,
 ('_108', False): 36,
 ('_109', False): 34,
 ('_10', False): 35,
 ('_110', False): 36,
 ('_111', False): 37,
 ('_112', False): 37,
 ('_113', False): 33,
 ('_114', False): 36,
 ('_115', False): 37,
 ('_116', False): 37,
 ('_117', False): 37,
 ('_118', False): 34,
 ('_119', False): 36,
 ('_11', False): 30,
 ('_120', False): 34,
 ('_121', False): 35,
 ('

In [96]:
freqs = count_utts({}, utts, ys)
lookup(freqs, "be", True)
for k, v in freqs.items():
    if "be" in k:
        print(f"{k}:{v}")

In [97]:
def train_naive_bayes(freqs, train_x, train_y):
    """
    Input:
        freqs: (단어, 레이블) 튜플을 해당 단어 빈도에 대응시킨 사전 객체
        train_x: 발화들의 목록
        train_y: 발화의 레이블(0 또는 1)의 목록
    Output:
        logprior: 로그 사전확률
        loglikelihood: 단순 베이즈 방정식의 로그우도(로그가능도)
    """
    loglikelihood = {}
    logprior = 0

    # 어휘 사전의 고유 단어 수인 V를 계산한다.
    vocab = set([pair[0] for pair in freqs.keys()])
    V = len(vocab)

    # 근정적인 단어 개수 N_pos와 부정적인 N_neg를 초기화한다.
    N_pos = N_neg = 0
    for pair in freqs.keys():
        # 감성 분류를 나타내는 레이블이 양수(0보다 큼)인 경우
        if pair[1] > 0:
            # 긍정적인 단어 튜플(단어, 레이블)의 개수를 증가시킨다.
            N_pos += lookup(freqs, pair[0], True)

        # 그렇지 않으면 레이블은 음수이다.
        else:
            # 이 경우 부정적인 단어 튜플의 개수를 증가시킨다.
            N_neg += lookup(freqs, pair[0], False)

    # 전체 문서 개수인 D를 계산한다.
    D = len(train_y)

    # 긍정적인 문서의 개수를 계산한다.
    D_pos = sum(train_y)

    # 부정적인 문서의 개수를 계산한다.
    D_neg = D - D_pos

    # 로그 사전확률을 계산한다.
    logprior = np.log(D_pos) - np.log(D_neg)

    # 어휘의 각 단어에 대해 연산
    for word in vocab:
        freq_pos = lookup(freqs, word, 1)
        freq_neg = lookup(freqs, word, 0)

        # 주어진 단어가 긍정적일 확률과 부정적인 확률을 계산한다.
        p_w_pos = (freq_pos + 1) / (N_pos + V)
        p_w_neg = (freq_pos + 1) / (N_neg + V)

        # 단어의 로그우도를 계산한다.
        loglikelihood[word] = np.log(p_w_pos / p_w_neg)

    return logprior, loglikelihood

In [109]:
def naive_bayes_predict(utt, logprior, loglikelihood):
    """
    Input:
        utt: a string
        logprior: a number
        loglikelihood: a dictionary of words mapping to numbers
    Output:
        p: the sum of all the logliklihoods + logprior
    """
    # process the utt to get a list of words
    word_l = process_utt(utt)

    # initialize probability to zero
    p = 0

    # add the logprior
    p += logprior

    for word in word_l:
        # check if the word exists in the loglikelihood dictionary
        if word in loglikelihood:
            # add the log likelihood of that word to the probability
            p += loglikelihood[word]

    return p

In [110]:
def test_naive_bayes(test_x, test_y, logprior, loglikelihood):
    """
    Input:
        test_x: A list of utts
        test_y: the corresponding labels for the list of utts
        logprior: the logprior
        loglikelihood: a dictionary with the loglikelihoods for each word
    Output:
        accuracy: (# of utts classified correctly)/(total # of utts)
    """
    accuracy = 0  # return this properly

    y_hats = []
    for utt in test_x:
        # if the prediction is > 0
        if naive_bayes_predict(utt, logprior, loglikelihood) > 0:
            # the predicted class is 1
            y_hat_i = 1
        else:
            # otherwise the predicted class is 0
            y_hat_i = 0

        # append the predicted class to the list y_hats
        y_hats.append(y_hat_i)

    # error = avg of the abs vals of the diffs between y_hats and test_y
    error = sum(
        [abs(y_hat - test) for y_hat, test in zip(y_hats, test_y)]
    ) / len(y_hats)

    # Accuracy is 1 minus the error
    accuracy = 1 - error

    return accuracy

In [111]:
if __name__ == "__main__":
    logprior, loglikelihood = train_naive_bayes(freqs, utts, ys)
    print(logprior)
    print(len(loglikelihood))

    my_utt = "To be or not to be, that is the question."
    p = naive_bayes_predict(my_utt, logprior, loglikelihood)
    print("The expected output is", p)

    print(
        "Naive Bayes accuracy = %0.4f"
        % (test_naive_bayes(utts, ys, logprior, loglikelihood))
    )

/tmp/ipykernel_4426/3032017796.py:41: RuntimeWarning: divide by zero encountered in log
  logprior = np.log(D_pos) - np.log(D_neg)


-inf
368
The expected output is -inf
Naive Bayes accuracy = 1.0000


## 2-3 생성형 은닉 마르코프 모델

In [112]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [113]:
import re
import random
from nltk.tokenize import word_tokenize
from collections import defaultdict, deque

In [116]:
class MarkovChain:
    def __init__(self):
        self.lookup_dict = defaultdict(list)
        self._seeded = False
        self.__seed_me()

    def __seed_me(self, rand_seed=None):
        if self._seeded is not True:
            try:
                if rand_seed is not None:
                    random.seed(rand_seed)
                else:
                    random.seed()
                self._seeded = True
            except NotImplementedError:
                self._seeded = False

    def add_document(self, str):
        preprocessed_list = self._preprocess(str)
        pairs = self.__generate_tuple_keys(preprocessed_list)
        for pair in pairs:
            self.lookup_dict[pair[0]].append(pair[1])

    def _preprocess(self, str):
        cleaned = re.sub(r"\W+", " ", str).lower()
        tokenized = word_tokenize(cleaned)
        return tokenized

    def __generate_tuple_keys(self, data):
        if len(data) < 1:
            return

        for i in range(len(data) - 1):
            yield [data[i], data[i+1]]

    def generate_text(self, max_length=50):
        context = deque()
        output = []
        if len(self.lookup_dict) > 0:
            self.__seed_me(rand_seed=len(self.lookup_dict))
            chain_head = [list(self.lookup_dict)[0]]
            context.extend(chain_head)

            while len(output) < (max_length - 1):
                next_choices = self.lookup_dict[context[-1]]
                if len(next_choices) > 0:
                    next_word = random.choice(next_choices)
                    context.append(next_word)
                    output.append(context.popleft())
                else:
                    break
            output.extend(list(context))
        return " ".join(output)

In [119]:
if __name__ == "__main__":
    with open("/content/hamlet.txt", "r", encoding="utf-8") as f:
        text = f.read()
    HMM = MarkovChain()
    HMM.add_document(text)

    print(HMM.generate_text(max_length=25))

the offence in creating the card 24 and maggots in conformity with most constantly _ham _ my place as you how tropically 82 _tropically _


## 2-4 생성형 cbow모델과  embeddings의 시각화

In [120]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
